In [1]:
import mlflow

mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("brain-tumor-detection")

print(mlflow.get_tracking_uri())

2026/10/31 12:22:17 INFO mlflow.tracking.fluent: Experiment with name 'brain-tumor-detection' does not exist. Creating a new experiment.


http://mlflow:5000


In [2]:
import pandas as pd

run_folder = "/app/models/detection/yolov8s_run1"

with mlflow.start_run(run_name="yolov8s_run1"):

    mlflow.log_param("model", "yolov8s")
    mlflow.log_param("epochs", 50)
    mlflow.log_param("batch", 16)
    mlflow.log_param("imgsz", 640)
    mlflow.log_param("patience", 10)
    mlflow.log_param("seed", 42)
    mlflow.log_param("hsv_v", 0.3)
    mlflow.log_param("degrees", 10)
    mlflow.log_param("scale", 0.3)
    mlflow.log_param("fliplr", 0.5)
    mlflow.log_param("mosaic", 1.0)

    df = pd.read_csv(run_folder + "/results.csv")
    df.columns = df.columns.str.strip()

    for i in range(len(df)):
        epoch = int(df["epoch"][i])
        for column in df.columns:
            if column == "epoch" or column.startswith("lr/"):
                continue
            name = column.replace("(B)", "")
            mlflow.log_metric(name, float(df[column][i]), step=epoch)

    mlflow.log_metric("test_precision", 0.931)
    mlflow.log_metric("test_recall", 0.917)
    mlflow.log_metric("test_mAP50", 0.950)
    mlflow.log_metric("test_mAP50_95", 0.753)

    mlflow.log_artifact(run_folder + "/weights/best.pt", "model")
    mlflow.log_artifact(run_folder + "/results.csv", "results")
    mlflow.log_artifact("/app/data/processed/detection/data.yaml", "config")
    mlflow.log_artifact("/app/models/detection/test_eval/confusion_matrix.png", "plots")

print("Logged")

2026/10/31 12:23:03 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet

2026/10/31 12:23:40 INFO mlflow.tracking._tracking_service.client: 🏃 View run yolov8s_run1 at: http://ml

Logged
